# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [2]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 22)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [3]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,17445914.41,3379.68,1167.38,26602
Audio,3949,9234318.00,2338.39,1073.00,15078
Mobilier,3936,8363145.58,2124.78,717.48,18274
Accessoire,3870,7575740.68,1957.56,834.65,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [4]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  1.744591e+07               0.0026
1      Mobilier   140.960  8.363146e+06               0.0017
2         Audio  2073.360  9.234318e+06               0.0225
3  Informatique  2281.000  1.744591e+07               0.0131
4      Mobilier  1904.490  8.363146e+06               0.0228

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [5]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [6]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

# TODO
revenue_crosstab = sales.pivot_table(index='categorie', columns='canal', values='montant',
                                     aggfunc='sum', margins=True, margins_name='Total').round(0)
revenue_crosstab

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2168517.0,2311010.0,3096213.0,7575741.0
Audio,2826938.0,861985.0,5545395.0,9234318.0
Informatique,6269923.0,4832327.0,6343664.0,17445914.0
Mobilier,3379260.0,2469037.0,2514848.0,8363146.0
Total,14644639.0,10474359.0,17500121.0,42619119.0


In [7]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

# TODO
customer_summary = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
).round(2)

print('Clients ayant commandé :', len(customer_summary))
print()
print('Dix meilleurs clients par CA :')
customer_summary.nlargest(10, 'ca_total')

Clients ayant commandé : 3051

Dix meilleurs clients par CA :


,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00696,7,1110180.19,158597.17,2023-02-12 00:00:00,2024-11-19 00:00:00,3
C02327,4,907834.48,226958.62,2023-10-31 00:00:00,2024-12-28 14:36:00,2
C00279,10,876999.45,87699.95,2023-01-24 00:00:00,2024-11-05 00:00:00,4
C02631,10,788266.88,78826.69,2023-01-12 00:00:00,2024-12-19 00:00:00,4
C00941,7,783179.31,111882.76,2023-01-29 00:00:00,2024-09-15 00:00:00,3
C00355,13,779167.47,59935.96,2023-02-12 11:09:00,2024-09-25 00:00:00,4
C02101,6,750743.90,125123.98,2023-05-01 11:00:00,2024-02-03 00:00:00,3
C00806,5,707386.06,141477.21,2023-03-05 00:00:00,2024-08-26 00:00:00,2
C02285,7,687164.51,98166.36,2023-05-19 00:00:00,2024-12-27 00:00:00,2


In [8]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

# TODO
median_by_category = sales.groupby('categorie')['montant'].transform('median')
sales['depasse_median_50pct'] = sales['montant'] > 1.5 * median_by_category

print('Commandes signalées :', sales['depasse_median_50pct'].sum(),
      f"({sales['depasse_median_50pct'].mean():.1%} des commandes)")
print()
print('Part signalée par catégorie :')
print(sales.groupby('categorie')['depasse_median_50pct'].mean().round(3))
# Les lignes sans catégorie (produit inconnu P9999) ont une médiane NaN : la comparaison
# renvoie False, elles ne sont donc jamais signalées.

Commandes signalées : 5662 (33.3% des commandes)

Part signalée par catégorie :
categorie
Accessoire      0.290
Audio           0.344
Informatique    0.348
Mobilier        0.352
Name: depasse_median_50pct, dtype: float64


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [9]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [10]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [11]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [12]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

# TODO
# ventes_transformees contient déjà les colonnes produits (séance 4) : on repart des
# seules colonnes de vente pour que la jointure soit faite ici, et pas en double.
product_columns = [column for column in products.columns if column != 'id_produit']
analytic = sales.drop(columns=product_columns, errors='ignore')
row_counts = {'ventes': len(analytic)}

# 1. Produits : clé unique à droite -> many_to_one
analytic = analytic.merge(products, on='id_produit', how='left', validate='many_to_one')
row_counts['+ produits'] = len(analytic)

# 2. Clients
analytic = analytic.merge(customers, on='id_client', how='left', validate='many_to_one')
row_counts['+ clients'] = len(analytic)

# 3. Magasins : 'ville' existe aussi côté clients, on suffixe pour lever l'ambiguïté
analytic = analytic.merge(stores, on='id_magasin', how='left', validate='many_to_one',
                          suffixes=('_client', '_magasin'))
row_counts['+ magasins'] = len(analytic)

for step, n_rows in row_counts.items():
    print(f'{step:<12} {n_rows} lignes')
assert len(set(row_counts.values())) == 1, 'une jointure a modifié le nombre de lignes'

# Clés orphelines
print()
print('Produits orphelins :', analytic['libelle'].isna().sum(),
      analytic.loc[analytic['libelle'].isna(), 'id_produit'].unique())
print('Clients orphelins  :', analytic['segment'].isna().sum())
print('Magasins orphelins :', analytic['nom_magasin'].isna().sum())

# Choix : les ventes du produit inconnu P9999 sont des ventes réelles (montant renseigné).
# Les supprimer ferait disparaître du chiffre d'affaires ; on les CONSERVE et on les
# rattache à un membre « inconnu » explicite, pratique standard en BI qui permet de
# les filtrer ou de les signaler, plutôt que de laisser des cases vides.
unknown_product = analytic['libelle'].isna()
analytic.loc[unknown_product, ['libelle', 'categorie', 'sous_categorie']] = 'inconnu'
print('Lignes rattachées au produit inconnu :', unknown_product.sum())
print('Forme finale :', analytic.shape)

ventes       16992 lignes
+ produits   16992 lignes
+ clients    16992 lignes
+ magasins   16992 lignes

Produits orphelins : 75 <ArrowStringArray>
['P9999']
Length: 1, dtype: str
Clients orphelins  : 0
Magasins orphelins : 0
Lignes rattachées au produit inconnu : 75
Forme finale : (16992, 33)


In [13]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    ...  # TODO
    """Jointure contrôlée : vérifie le nombre de lignes et signale les clés non appariées."""
    how = kwargs.get('how', 'inner')
    merged = left.merge(right, indicator='_appariement', **kwargs)

    if how == 'left' and len(merged) != len(left):
        raise ValueError(f'left join : {len(left)} lignes en entrée, {len(merged)} en sortie '
                         '(clé non unique à droite ?)')

    unmatched_rate = (merged['_appariement'] == 'left_only').mean()
    print(f'safe_merge ({how}) : {len(left)} -> {len(merged)} lignes, '
          f'clés non appariées : {unmatched_rate:.2%}')
    return merged.drop(columns='_appariement')


# Démonstration : jointure saine, puis jointure sur une clé dupliquée
check = safe_merge(sales[['id_commande', 'id_produit']], products, on='id_produit', how='left')
try:
    safe_merge(sales[['id_commande', 'id_client']], customers_with_duplicates,
               on='id_client', how='left')
except ValueError as error:
    print('ValueError :', error)

safe_merge (left) : 16992 -> 16992 lignes, clés non appariées : 0.44%
ValueError : left join : 16992 lignes en entrée, 17287 en sortie (clé non unique à droite ?)


---
## Atelier 6.3 — Formats long et large (35 min)

In [14]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,377700.0,290306.0,340485.0,660904.0,528372.0,345662.0,341055.0,1839610.0,486776.0,1024218.0,896564.0,444090.0
Audio,1202719.0,662225.0,393742.0,1094554.0,445550.0,423529.0,590389.0,1805403.0,424471.0,411295.0,1132486.0,647955.0
Informatique,1803339.0,1589647.0,3416002.0,689139.0,1626278.0,639193.0,1839227.0,734560.0,1694766.0,702179.0,711393.0,2000191.0
Mobilier,905905.0,315378.0,306098.0,1129765.0,1343534.0,623024.0,935075.0,460363.0,782626.0,386601.0,706635.0,468141.0


In [15]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,377700.0
1,Audio,mois_01,1202719.0
2,Informatique,mois_01,1803339.0
3,Mobilier,mois_01,905905.0
4,Accessoire,mois_02,290306.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [16]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

# TODO
wide_again = long.set_index(['categorie', 'mois'])['ca'].unstack('mois')
wide_again.columns.name = None

pd.testing.assert_frame_equal(wide_again, wide)
print('OK — le tableau large est retrouvé à l\'identique')
wide_again.head()

OK — le tableau large est retrouvé à l'identique


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,377700.0,290306.0,340485.0,660904.0,528372.0,345662.0,341055.0,1839610.0,486776.0,1024218.0,896564.0,444090.0
Audio,1202719.0,662225.0,393742.0,1094554.0,445550.0,423529.0,590389.0,1805403.0,424471.0,411295.0,1132486.0,647955.0
Informatique,1803339.0,1589647.0,3416002.0,689139.0,1626278.0,639193.0,1839227.0,734560.0,1694766.0,702179.0,711393.0,2000191.0
Mobilier,905905.0,315378.0,306098.0,1129765.0,1343534.0,623024.0,935075.0,460363.0,782626.0,386601.0,706635.0,468141.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [17]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [18]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

# TODO
STAR_DIR = PROCESSED_DIR / 'star'
STAR_DIR.mkdir(parents=True, exist_ok=True)

# Table de faits : une ligne par commande, clés étrangères + mesures.
# montant      : montant brut, quantité x prix unitaire ;
# montant_net  : après déduction de la remise (ce que le client paie réellement).
fact_sales = pd.DataFrame({
    'cle_date': sales['date_commande'].dt.strftime('%Y%m%d').astype(int),
    'id_client': sales['id_client'],
    'id_produit': sales['id_produit'],
    'id_magasin': sales['id_magasin'],
    'quantite': sales['quantite'],
    'prix_unitaire': sales['prix_unitaire'],
    'remise_pct': sales['remise_pct'].fillna(0),
})
fact_sales['montant'] = (fact_sales['quantite'] * fact_sales['prix_unitaire']).round(2)
fact_sales['montant_net'] = (fact_sales['montant'] * (1 - fact_sales['remise_pct'] / 100)).round(2)
fact_sales['statut'] = sales['statut'].values

# Dimensions, dédoublonnées sur leur clé primaire
dim_customer = customers.drop_duplicates('id_client').copy()
dim_customer['ville'] = dim_customer['ville'].str.strip().str.title()   # attribut descriptif

dim_product = products.drop_duplicates('id_produit')
# Membre « inconnu » pour les ventes orphelines (voir Q4) : elles restent dans les faits
missing_products = sorted(set(fact_sales['id_produit']) - set(dim_product['id_produit']))
dim_product = pd.concat([dim_product, pd.DataFrame({
    'id_produit': missing_products, 'libelle': 'inconnu',
    'categorie': 'inconnu', 'sous_categorie': 'inconnu', 'cout_achat': np.nan,
})], ignore_index=True)

dim_store = stores.drop_duplicates('id_magasin')

star_tables = {'fact_sales': fact_sales, 'dim_date': dim_date, 'dim_customer': dim_customer,
               'dim_product': dim_product, 'dim_store': dim_store}
for name, table in star_tables.items():
    table.to_parquet(STAR_DIR / f'{name}.parquet', index=False)
    print(f'écrit : star/{name}.parquet {table.shape}')

# Vérification rapide de l'intégrité référentielle
print()
for key, dimension in [('cle_date', dim_date), ('id_client', dim_customer),
                       ('id_produit', dim_product), ('id_magasin', dim_store)]:
    n_orphans = (~fact_sales[key].isin(dimension[key])).sum()
    print(f'{key:<11} clés orphelines : {n_orphans}')

écrit : star/fact_sales.parquet (16992, 10)
écrit : star/dim_date.parquet (730, 10)
écrit : star/dim_customer.parquet (3060, 5)
écrit : star/dim_product.parquet (40, 5)
écrit : star/dim_store.parquet (9, 5)

cle_date    clés orphelines : 0
id_client   clés orphelines : 0
id_produit  clés orphelines : 0
id_magasin  clés orphelines : 0


In [19]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""
    ...  # TODO
    problems = {}
    for key, dimension in dimensions.items():
        orphan_keys = set(fact_table[key].dropna()) - set(dimension[key])
        n_orphan_rows = int(fact_table[key].isin(orphan_keys).sum())
        n_missing_keys = int(fact_table[key].isna().sum())
        print(f'{key:<11} clés orphelines : {len(orphan_keys):>3} '
              f'({n_orphan_rows} lignes) | clés absentes : {n_missing_keys}')
        if orphan_keys or n_missing_keys:
            problems[key] = sorted(orphan_keys)[:10]
    if problems:
        raise ValueError(f'Intégrité référentielle violée : {problems}')
    print('OK — intégrité référentielle respectée')


check_referential_integrity(fact_sales, {'cle_date': dim_date, 'id_client': dim_customer,
                                         'id_produit': dim_product, 'id_magasin': dim_store})

# Le contrôle échoue bien bruyamment sur une dimension incomplète :
try:
    check_referential_integrity(fact_sales, {'id_produit': products})
except ValueError as error:
    print('ValueError :', error)

cle_date    clés orphelines :   0 (0 lignes) | clés absentes : 0
id_client   clés orphelines :   0 (0 lignes) | clés absentes : 0
id_produit  clés orphelines :   0 (0 lignes) | clés absentes : 0
id_magasin  clés orphelines :   0 (0 lignes) | clés absentes : 0
OK — intégrité référentielle respectée


id_produit  clés orphelines :   1 (75 lignes) | clés absentes : 0
ValueError : Intégrité référentielle violée : {'id_produit': ['P9999']}


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.